In [1]:
import pandas as pd

In [34]:
time_col = "S[table-format=1.3\\,s, round-mode=places, round-precision=3]"


def format_name(name):
    if name.endswith("__MOD_PCG"):
        name = name[:-9]
    return f"\\verb|{name}|"


def generate_latex(df):
    df = df[["config", "setup_time", "solve_time"]].sort_values("solve_time").copy()
    df = df[~df["config"].str.contains("PMIS")]
    df = df[~df["config"].str.contains("D2")]
    df.columns = ["{Config}", "{Setup time}", "{Solution time}"]
    df.set_index("{Config}", inplace=True)

    df_latex = (
        df.style.format_index(format_name, axis=0)
        .format(lambda t: f"{t:.6f}\\, \\si{{\\second}}")
        .to_latex(hrules=True, column_format=f"l{time_col}{time_col}")
    )

    latex_lines = df_latex.split("\n")
    first_header_line = latex_lines.index("\\toprule") + 1
    last_header_line = latex_lines.index("\\midrule") - 1
    column_names = [None] * len(latex_lines[first_header_line].split("&"))
    for line in latex_lines[first_header_line : last_header_line + 1]:
        line_col_names = [x.strip().removesuffix("\\\\") for x in line.split("&")]
        for i, col_name in enumerate(line_col_names):
            if col_name:
                column_names[i] = col_name
    single_header_line = " & ".join(column_names) + " \\\\"
    hacked_pivot_latex = "\n".join(
        latex_lines[:first_header_line]
        + [single_header_line]
        + latex_lines[last_header_line + 1 :]
    )

    return hacked_pivot_latex

In [35]:
df = pd.read_csv("../results/amgx_2d.csv")
with open("../docs/tables/amgx_2d.tex", "w") as f:
    f.write(generate_latex(df))

In [36]:
df = pd.read_csv("../results/amgx_3d.csv")
with open("../docs/tables/amgx_3d.tex", "w") as f:
    f.write(generate_latex(df))